<a href="https://colab.research.google.com/github/manmathbh/LLM/blob/main/Implementing_data_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1 : Creating Tokens


In [1]:
with open("/content/drive/MyDrive/Building LLM/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

print("Total chars: ", len(raw_text))
print(raw_text[:99])

Total chars:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
import re
text = "hello, world. This is Manmath."
result = re.split(r'(\s)', text)
print(result)

['hello,', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'Manmath.']


In [3]:
result = re.split(r'([,.]|\s)', text)
print(result)

['hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ' ', 'is', ' ', 'Manmath', '.', '']


In [4]:
result = [item for item in result if item.strip()]
print(result)

['hello', ',', 'world', '.', 'This', 'is', 'Manmath', '.']


In [5]:
text = "Hello, guys. Is this-- a ai?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'guys', '.', 'Is', 'this', '--', 'a', 'ai', '?']


In [6]:
result = [it for it in result if it.strip()]
print(result)

['Hello', ',', 'guys', '.', 'Is', 'this', '--', 'a', 'ai', '?']


In [7]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [it.strip() for it in preprocessed if it.strip()]
print(preprocessed[:99])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter']


In [8]:
print(len(preprocessed))

4690


STEP 2: CREATING TOKEN IDS


In [9]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [10]:
vocab = {token:integer for integer, token in enumerate(all_words)}


In [11]:
for i, it in enumerate(vocab.items()):
  print(it)
  if i >= 50:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [12]:
class SimpleTokenizer1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [13]:
tokenizer = SimpleTokenizer1(vocab)

text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [14]:
text1 = tokenizer.decode(ids)
print(text1)

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [15]:
#s = "hi, Manmath is here, I'll heading this parliament!!"
#print(tokenizer.encode(s))
#it will give error: bcz the words in the given text should be present in vocab

A basic tokenizer crashes on new words and gets confused when unrelated documents are merged together. You fix this by adding two special "meta-tags" to your vocabulary:

<|unk|> (Unknown): Replaces any word the tokenizer doesn't recognize so the program doesn't crash.

<|endoftext|> (End of Text): Placed between separate documents so the AI knows the context has reset and the texts are unrelated.

In [16]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [17]:
print(len(vocab))

1132


In [18]:
for i,it in enumerate(list(vocab.items())[-5:]):
  print(it)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


Step 1: Replace unknown words by <|unk|> tokens

Step 2: Replace spaces before the specified punctuations

In [19]:
class SimpleTokenizer2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [20]:
manmath_tokenizer = SimpleTokenizer2(vocab)
text1 = "Hello everyone, I'm Manmath!"
text2 = "I'll heading this parliament."
text = " <|endoftext|>".join((text1, text2))
print(text)

Hello everyone, I'm Manmath! <|endoftext|>I'll heading this parliament.


In [21]:
manmath_tokenizer.encode(text)

[1131, 1131, 5, 53, 2, 1131, 1131, 0, 1131, 2, 637, 1131, 999, 1131, 7]

In [22]:
manmath_tokenizer.decode(manmath_tokenizer.encode(text))

"<|unk|> <|unk|>, I' <|unk|> <|unk|>! <|unk|>' ll <|unk|> this <|unk|>."

## BYTE PAIR ENCODING

# BPE Tokenizer

In [23]:
! pip3 install tiktoken

In [24]:
import importlib
import tiktoken

print("tiktoken vers: ", importlib.metadata.version("tiktoken"))

tiktoken vers:  0.12.0


In [25]:
tokenizer = tiktoken.get_encoding("gpt2")

In [26]:
text = (
    "Good morning ladies and gentlelmen, my question to you is, how can we protect our national dignity on international level??"
)

integers = tokenizer.encode(text, allowed_special = {"<|endoftext|>"})

print(integers)

[10248, 3329, 17308, 290, 10296, 75, 3653, 11, 616, 1808, 284, 345, 318, 11, 703, 460, 356, 1805, 674, 2260, 16247, 319, 3230, 1241, 3548]


In [27]:
strings = tokenizer.decode(integers)
print(strings)

Good morning ladies and gentlelmen, my question to you is, how can we protect our national dignity on international level??


In [28]:
integ = tokenizer.encode("Manmath Balaji Hatte")
print(integ)

strings = tokenizer.decode(integ)
print(strings)

[5124, 11018, 8528, 26436, 10983, 660]
Manmath Balaji Hatte


# CREATING INPUT-TARGET PAIRS

In [29]:
with open("/content/drive/MyDrive/Building LLM/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [30]:
enc_sample = enc_text[50:]

In [31]:
context_size = 4 #slide window len
#The input x is the first 4 tokens [1, 2, 3, 4], and the target y is the next 4 tokens [2, 3, 4, 5]

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


In [32]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [33]:
#decoding
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


# Implementing data loader

Step 1: Tokenize the entire text

Step 2: Use a sliding window to chunk the book into overlapping sequences of max_length

Step 3: Return the total number of rows in the dataset

Step 4: Return a single row from the dataset

In [34]:
from torch.utils.data import Dataset, DataLoader


class GPTDataset1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        #sliding window
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

Step 1: Initialize the tokenizer

Step 2: Create dataset

Step 3: drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training

Step 4: The number of CPU processes to use for preprocessing

In [35]:
def create_dataloader_1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDataset1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [38]:
with open("/content/drive/MyDrive/Building LLM/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [40]:
import torch
# print("PyTorch version:", torch.__version__)  ===>    PyTorch version: 2.10.0+cpu
dataloader = create_dataloader_1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
batch1 = next(data_iter)
print(batch1)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [43]:
# second batch of len 4
batch2 = next(data_iter)
print(batch2)

batch3 = next(data_iter)
print(batch3)

[tensor([[1464, 1807, 3619,  402]]), tensor([[1807, 3619,  402,  271]])]
[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [44]:
dataloader = create_dataloader_1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
